## About the Model : Mixture of Experts (MoE)

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 3px;
    font-color :  #581845  ;        
    border: 1px solid #FF5733 ;">  
    
Mixtral 8x7B, often referred to as a "miniature GPT-4," leverages a Mixture of Experts (MoE) architecture with eight individual experts, is a 45B parameter model. This architectural decision is noteworthy because it allows only two experts to participate in the inference process for each token, signifying a move towards more streamlined and targeted AI processing.

A standout feature of Mixtral is its capacity to handle an extensive context of 32,000 tokens, offering a wide-ranging scope for tackling intricate tasks. Moreover, the model's multilingual support extends to English, French, Italian, German, and Spanish, making it a versatile tool for a diverse global developer community.

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FFFFFF;
    border-radius: 20px;
    font-color :  #581845  ;        
    border: 4px solid #000000 ;"> 
    
Don't forget to upvote 👆 if you find it useful :) 

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 1px solid #FF5733 ;"> 
    
- Installing the relevant libraries from the github repository itself : [POV : There are methods like e.g. 4-bit peft adapter merge_and_unload() which are still in the beta version, else every functionality which is in the released version behaves the same way]

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

## Quantization

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 
    
- If you're new to quantization,start by reading [this](https://en.wikibooks.org/wiki/A-level_Computing/AQA/Paper_2/Fundamentals_of_data_representation/Floating_point_numbers#:~:text=In%20decimal%2C%20very%20large%20numbers,be%20used%20for%20binary%20numbers.) page.
    
- The quantization method is based on the paper QLoRA paper whose abstract follows as :
    >*We present QLoRA, an efficient finetuning approach that reduces memory usage enough to finetune a 65B parameter model on a single 48GB GPU while preserving full 16-bit finetuning task performance. QLoRA backpropagates gradients through a frozen, 4-bit quantized pretrained language model into Low Rank Adapters~(LoRA). Our best model family, which we name Guanaco, outperforms all previous openly released models on the Vicuna benchmark, reaching 99.3% of the performance level of ChatGPT while only requiring 24 hours of finetuning on a single GPU. QLoRA introduces a number of innovations to save memory without sacrificing performance: (a) 4-bit NormalFloat (NF4), a new data type that is information theoretically optimal for normally distributed weights (b) double quantization to reduce the average memory footprint by quantizing the quantization constants, and (c) paged optimizers to manage memory spikes. We use QLoRA to finetune more than 1,000 models, providing a detailed analysis of instruction following and chatbot performance across 8 instruction datasets, multiple model types (LLaMA, T5), and model scales that would be infeasible to run with regular finetuning (e.g. 33B and 65B parameter models). Our results show that QLoRA finetuning on a small high-quality dataset leads to state-of-the-art results, even when using smaller models than the previous SoTA. We provide a detailed analysis of chatbot performance based on both human and GPT-4 evaluations showing that GPT-4 evaluations are a cheap and reasonable alternative to human evaluation. Furthermore, we find that current chatbot benchmarks are not trustworthy to accurately evaluate the performance levels of chatbots. A lemon-picked analysis demonstrates where Guanaco fails compared to ChatGPT. We release all of our models and code, including CUDA kernels for 4-bit training*
    
- Bitsandbytes makes it easy to quantize and finetune the model with lesser memory requirements

In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    #model_name = "/kaggle/input/qwen-3/transformers/30b-a3b/1",
    model_name =  'allenai/OLMoE-1B-7B-0125-Instruct',
    device_map='auto', 
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 
    
- `load_in_4bit` : bitsandbytes stores weights in 4-bits
- `bnb_4bit_quant_type= "nf4"` : normalized float 4 (as per the QLoRA paper,using NF4 quantization is recommended for better performance
- `bnb_4bit_compute_dtype= torch.bfloat16` : 
<p><img src="https://storage.googleapis.com/gweb-cloudblog-publish/images/Three_floating-point_formats.max-700x700.png" height="600" width="600" style="object-fit: cover;"></p>
<p style="text-align:justify;">
</p>  
The dynamic range of bfloat16 and float32 are equivalent. However, bfloat16 takes up half the memory space
- `bnb_4bit_use_double_quant= True` :  Uses a second quantization after the first one to save an additional 0.4 bits per parameter
- `llm_int8_enable_fp32_cpu_offload = True` : If you want to split your model in different parts and run some parts in int8 on GPU and some parts in fp32 on CPU, you can use this flag. This is useful for offloading large models such as google/flan-t5-xxl. Note that the int8 operations will not be run on CPU.


## Get Lo/MoE Model

* 建议，只训练OlmoeDecoderLayer，不训练OlmoeSparseMoeBlock
* gate层‌：(gate): Linear4bit(in_features=2048, out_features=64, bias=False)
此为路由层的核心，输入2048维特征，输出64维权重（对应64个专家），通过Softmax生成专家选择概率。
* experts列表‌（第12-19行）：包含64个OlmoeMLP专家模块，由路由权重动态激活部分专家。

In [3]:
model = FastModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 
    
- `device_map="auto"` : pass "auto" to get a device map that will be automatically inferred).Manually setting a device once the model has been loaded with device_map is not recommended when using accelerate. So any device assignment call to the model, or to any model’s submodules should be avoided after that line - unless you know what you are doing
- Set `trust_remote_code=True` to use a model with custom code 

## Get the Data

In [5]:
from datasets import load_dataset
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [6]:
import pandas as pd
print(reasoning_dataset)
reasoning_dataset.to_pandas()

Dataset({
    features: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'],
    num_rows: 19252
})


,expected_answer,problem_type,problem_source,generation_model,pass_rate_72b_tir,problem,generated_solution,inference_mode
0,14,has_answer_extracted,aops_c4_high_school_math,DeepSeek-R1,0.96875,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ...",cot
1,convergent,has_answer_extracted,aops_c7_college_math,DeepSeek-R1,0.96875,Let \( \sum a_n \) be a convergent series with...,"<think>\nOkay, so I have this problem here whe...",cot
2,\(\frac{\pi \ln a}{2a}\),has_answer_extracted,aops_c7_college_math,DeepSeek-R1,0.96875,"For \( a > 0 \), find the value of \( \int_0^{...","<think>\nOkay, so I need to solve the integral...",cot
3,\(+\infty\),has_answer_extracted,aops_c7_college_math,DeepSeek-R1,0.96875,Calculate $\lim_{n\to\infty}\sqrt[n]{n!}$.,"<think>\nOkay, so I need to find the limit as ...",cot
4,\( n = 1 \),has_answer_extracted,aops_c6_high_school_olympiads,DeepSeek-R1,0.96875,Find all positive integers \( n \) such that \...,"<think>\nOkay, so I need to find all positive ...",cot
...,...,...,...,...,...,...,...,...
19247,4,has_answer_extracted,aops_c4_high_school_math,DeepSeek-R1,0.96875,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ...",cot
19248,18,has_answer_extracted,aops_c4_high_school_math,DeepSeek-R1,0.96875,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n...",cot
19249,\( e^3 - 1 \),has_answer_extracted,aops_c7_college_math,DeepSeek-R1,0.96875,Evaluate the series \( \sum_{k=1}^{\infty}\fra...,"<think>\nOkay, so I need to evaluate the serie...",cot
19250,0.8960,has_answer_extracted,aops_c4_high_school_math,DeepSeek-R1,0.96875,Find the probability that the second blue resu...,"<think>\nOkay, so I need to find the probabili...",cot


<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 

- Python’s memory allocation and deallocation method is automatic. The user does not have to preallocate or deallocate memory.
- Invoking the garbage collector (using `gc.collect`) manually during the execution of a program can be a good idea for how to handle memory being consumed by reference cycles.

In [7]:
import pandas as pd
print(non_reasoning_dataset)
non_reasoning_dataset.to_pandas()

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 100000
})


,conversations,source,score
0,"[{'from': 'human', 'value': 'Explain what bool...",infini-instruct-top-500k,5.212621
1,"[{'from': 'human', 'value': 'Explain how recur...",infini-instruct-top-500k,5.157649
2,"[{'from': 'human', 'value': 'Explain what bool...",infini-instruct-top-500k,5.147540
3,"[{'from': 'human', 'value': 'Explain the conce...",infini-instruct-top-500k,5.053656
4,"[{'from': 'human', 'value': 'Print the reverse...",infini-instruct-top-500k,5.045648
...,...,...,...
99995,"[{'from': 'human', 'value': 'How can I create ...",infini-instruct-top-500k,3.735728
99996,"[{'from': 'human', 'value': 'How do I distingu...",WebInstructSub_axolotl,3.735725
99997,"[{'from': 'human', 'value': 'In a steady syste...",WebInstructSub_axolotl,3.735725
99998,"[{'from': 'system', 'value': 'Your role is to ...",systemchat-2.0-sharegpt,3.735710


In [8]:
def generate_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

In [9]:
reasoning_conversations = tokenizer.apply_chat_template(
    reasoning_dataset.map(generate_conversation, batched = True)["conversations"],
    tokenize = False,
)

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

In [10]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(non_reasoning_dataset)

non_reasoning_conversations = tokenizer.apply_chat_template(
    dataset["conversations"],
    tokenize = False,
)

Unsloth: Standardizing formats (num_proc=4):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [12]:
import pandas as pd

chat_percentage = 0.25
non_reasoning_subset = pd.Series(non_reasoning_conversations)
non_reasoning_subset = non_reasoning_subset.sample(
    int(len(reasoning_conversations)*(chat_percentage/(1 - chat_percentage))),
    random_state = 2407,
)

print(len(reasoning_conversations))
print(len(non_reasoning_subset))
print(len(non_reasoning_subset) / (len(non_reasoning_subset) + len(reasoning_conversations)))

19252
6417
0.2499902606256574


In [13]:
data = pd.concat([
    pd.Series(reasoning_conversations),
    pd.Series(non_reasoning_subset)
])
data.name = "text"

from datasets import Dataset
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = 3407)
pd.DataFrame(data)

,text
0,|||IP_ADDRESS|||<|user|>\nGiven $\sqrt{x^2+165...
1,|||IP_ADDRESS|||<|user|>\nLet \( \sum a_n \) b...
2,"|||IP_ADDRESS|||<|user|>\nFor \( a > 0 \), fin..."
3,|||IP_ADDRESS|||<|user|>\nCalculate $\lim_{n\t...
4,|||IP_ADDRESS|||<|user|>\nFind all positive in...
...,...
59440,|||IP_ADDRESS|||<|user|>\nConstruct a frequenc...
35212,|||IP_ADDRESS|||<|user|>\nWhat are the two pri...
18726,|||IP_ADDRESS|||<|user|>\nWhat measures can sc...
14793,|||IP_ADDRESS|||<|user|>\nWhy is a cavity with...


## Train Model

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 

#### Creating a prompt template alongwith the input for passing it to the model
- Here I'm passing an abstract to the model, and asking it to create a catchy title that maximises the clickbait. The prompt is in the format as desired by the mistral models (mentioned on [this](https://www.kaggle.com/models/mistral-ai/mixtral/frameworks/PyTorch/variations/8x7b-instruct-v0.1-hf/versions/1) page as well)

In [14]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/25669 [00:00<?, ? examples/s]

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 

#### To filter out any unnecessary warnings

In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 25,669 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 310,378,496 of 7,229,540,352 (4.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.449800
2,1.324100
3,1.322400
4,1.275200
5,1.230500
6,1.089000
7,1.053800
8,0.870800
9,0.949500
10,0.907500


## Model Inference

In [ ]:
messages = [
    {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 

- `do_sample=True` :  model.generate() method will use Sample Decoding
- `max_new_tokens=400` : Number of new tokens you want to generate
- `temperature=0.5` :  To increase the probability of probable tokens while reducing the one that is not : <p><img src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*41TqaBXrhIGU2V1JCEzU5Q.png" height="600" width="600" style="object-fit: cover;"></p>
<p style="text-align:justify;">
</p>   t is the temperature value.At temp=0.5, the most probable words like i, yeah, me, have more chance of being generated. At the same time, this also lowers the probability of the less probable ones, although this does not stop them from occurring.
- `top_p=0.95` :  Instead of considering all possible next words, top-p sampling only considers the smallest set of top words whose cumulative probability exceeds a certain threshold, p. A higher value of p means more words are considered, leading to more randomness in the generated text.
 <p><img src="https://api.wandb.ai/files/darek/images/projects/37727390/20e4f024.png" height="700" width="700" style="object-fit: cover;"></p>
<p style="text-align:justify;">

Learn more about it [here](https://towardsdatascience.com/decoding-strategies-that-you-need-to-know-for-response-generation-ba95ee0faadc)

In [ ]:
!nvidia-smi

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 

- On 2*T4 GPU vram occupied is almost 25 gb/ 30 gb
- Inference can be done easily.

<div class="anchor" id="top" style="
    margin-right: auto; 
    margin-left: auto;
    padding: 10px;
   font-size : 120%;
    background-color: #FEF2EF;
    border-radius: 2px;
    font-color :  #581845  ;        
    border: 2px solid #FF5733 ;"> 
    
    
### References : 

1. https://huggingface.co/docs/transformers/en/main_classes/quantization
2. https://huggingface.co/blog/4bit-transformers-bitsandbytes
3. https://huggingface.co/docs/transformers/en/custom_models
4. https://www.geeksforgeeks.org/garbage-collection-python/
5. https://www.kaggle.com/code/sangeek/pynvml-module-to-identify-and-monitor-gpu-usage
6. https://towardsdatascience.com/decoding-strategies-that-you-need-to-know-for-response-generation-ba95ee0faadc
7. https://wandb.ai/darek/llmapps/reports/A-Gentle-Introduction-to-LLM-APIs--Vmlldzo0NjM0MTMz
